# historical weighted three seeds

Run in order with the original historical V2 feature cache. This is a separate protocol from final topology.


# V2 Weighted GCN — Google Colab Free (EXACT Original Feature Cache + Auto-Resume)

This workflow reuses the recorded feature cache.

**RDKit features are NOT rebuilt on Colab.**  
Instead, the exact **(1705, 2249)** feature matrix is exported once from the original laptop environment and loaded in Colab.

This avoids the current RDKit-version mismatch that produced 2259 and then 2250 features.

## Files required in `MyDrive/DDI_V2/`
1. `clean_train_split.csv`
2. `clean_calibration_split.csv`
3. `clean_test_split.csv`
4. `clean_label_names.npy`
5. `exact_v2_feature_cache_2249.npz`
6. `exact_v2_feature_meta.json`

The two `exact_v2_feature_*` files are generated by:
`Export_Exact_V2_Features_Original_Laptop.ipynb`

Training still uses:
- 84 classes
- 2249 exact node features
- GCN 2249 → 384 → 384 → 192
- pair representation `[z1, z2, |z1-z2|, z1*z2]`
- weighted cross-entropy
- AdamW LR 7e-4
- batch size 512
- calibration Macro-F1 model selection
- auto-save after every completed epoch
- auto-resume from Google Drive


In [ ]:
# COLAB CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_FOLDER_NAME = "DDI_V2"
COLAB_PROJECT_DIR = os.path.join("/content/drive/MyDrive", PROJECT_FOLDER_NAME)
os.makedirs(COLAB_PROJECT_DIR, exist_ok=True)

print("Google Drive mounted ")
print("Project folder:", COLAB_PROJECT_DIR)


## GPU requirement

The next cell checks that an NVIDIA GPU is available.

If it says CUDA is unavailable, go to:

**Runtime → Change runtime type → GPU**

Then restart and run from the top. On Colab Free, GPU availability is not guaranteed at every moment.


In [ ]:
# COLAB STEP 4 — Imports
import os, time, json, copy, random, hashlib, platform, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

import rdkit
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Seed:", SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. In Colab choose Runtime → Change runtime type → GPU, "
        "then reconnect and run again."
    )

print("CUDA check PASSED ")

In [ ]:
# OPTIONAL — Upload the 4 required files from your computer into Google Drive
# Leave this False if you already placed the files in MyDrive/DDI_V2.

UPLOAD_FILES_FROM_COMPUTER = False

if UPLOAD_FILES_FROM_COMPUTER:
    from google.colab import files

    required_names = {
        "clean_train_split.csv",
        "clean_calibration_split.csv",
        "clean_test_split.csv",
        "clean_label_names.npy",
    }

    print("Select the 4 required files...")
    uploaded = files.upload()

    for name, data in uploaded.items():
        target = os.path.join(PROJECT_DIR, name)
        with open(target, "wb") as f:
            f.write(data)
        print("Saved:", target)

    missing_after_upload = [
        name for name in required_names
        if not os.path.exists(os.path.join(PROJECT_DIR, name))
    ]

    if missing_after_upload:
        print("Still missing:", missing_after_upload)
    else:
        print("All 4 required files are now in Google Drive ")
else:
    print("Upload helper OFF — using files already in Google Drive.")


In [ ]:
# COLAB STEP 7 — Check required files and environment
required_files = [TRAIN_PATH, CAL_PATH, TEST_PATH, LABELS_PATH]

for p in required_files:
    print(os.path.basename(p), "exists:", os.path.exists(p))

missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

environment_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "rdkit": rdkit.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

input_checksums = {os.path.basename(p): file_sha256(p) for p in required_files}

with open(os.path.join(SAVE_DIR, "environment.json"), "w", encoding="utf-8") as f:
    json.dump(environment_info, f, indent=2, ensure_ascii=False)

with open(os.path.join(SAVE_DIR, "input_checksums.json"), "w", encoding="utf-8") as f:
    json.dump(input_checksums, f, indent=2, ensure_ascii=False)

print(json.dumps(environment_info, indent=2))

In [ ]:
# COLAB STEP 8 — Load exact fixed splits
df_train = pd.read_csv(TRAIN_PATH)
df_cal   = pd.read_csv(CAL_PATH)
df_test  = pd.read_csv(TEST_PATH)

label_names = list(np.load(LABELS_PATH, allow_pickle=True))
num_classes = len(label_names)
label_to_id = {label: i for i, label in enumerate(label_names)}

for split_name, df_part in [("train", df_train), ("calibration", df_cal), ("test", df_test)]:
    required_cols = {"Drug1", "Drug2", "Label"}
    missing_cols = required_cols - set(df_part.columns)
    if missing_cols:
        raise ValueError(f"{split_name} missing columns: {missing_cols}")

    df_part["Drug1"] = df_part["Drug1"].astype(str)
    df_part["Drug2"] = df_part["Drug2"].astype(str)
    df_part["Label"] = df_part["Label"].astype(str)

    if "label_id" not in df_part.columns:
        df_part["label_id"] = df_part["Label"].map(label_to_id)

    if df_part["label_id"].isna().any():
        raise ValueError(f"Unmapped labels found in {split_name}")

    df_part["label_id"] = df_part["label_id"].astype(int)

print("Train:", len(df_train))
print("Calibration:", len(df_cal))
print("Test:", len(df_test))
print("Total:", len(df_train)+len(df_cal)+len(df_test))
print("Classes:", num_classes)

if STRICT_REPRO_CHECKS:
    assert len(df_train) == EXPECTED_TRAIN
    assert len(df_cal) == EXPECTED_CAL
    assert len(df_test) == EXPECTED_TEST
    assert len(df_train)+len(df_cal)+len(df_test) == EXPECTED_TOTAL
    assert num_classes == EXPECTED_CLASSES

print("Dataset validation PASSED ")

In [ ]:
# COLAB STEP 9 — Build drug index and TRAIN-ONLY graph
df_all = pd.concat([df_train, df_cal, df_test], axis=0).reset_index(drop=True)

all_drugs = pd.unique(pd.concat([df_all["Drug1"], df_all["Drug2"]], axis=0)).tolist()

drug_to_id = {drug: i for i, drug in enumerate(all_drugs)}
id_to_drug = {i: drug for drug, i in drug_to_id.items()}
num_nodes = len(all_drugs)

train_edges_df = df_train[["Drug1", "Drug2"]].drop_duplicates().reset_index(drop=True)

src = train_edges_df["Drug1"].map(drug_to_id).values
dst = train_edges_df["Drug2"].map(drug_to_id).values

edge_index = torch.tensor([src, dst], dtype=torch.long)
edge_index_rev = torch.tensor([dst, src], dtype=torch.long)
edge_index = torch.cat([edge_index, edge_index_rev], dim=1)

print("Unique drugs:", num_nodes)
print("Training unique pairs:", len(train_edges_df))
print("edge_index:", edge_index.shape)

if STRICT_REPRO_CHECKS:
    assert num_nodes == EXPECTED_UNIQUE_DRUGS

In [ ]:
import os
import zipfile
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/DDI_V2"

wanted = os.path.join(PROJECT_DIR, "exact_v2_feature_cache_2249.npz")
zipped = os.path.join(PROJECT_DIR, "exact_v2_feature_cache_2249.npz.zip")

if not os.path.exists(wanted) and os.path.exists(zipped):

    
    try:
        test = np.load(zipped, allow_pickle=True)

        if "x" in test.files and "all_drugs" in test.files:
            print("The file is an NPZ with .zip added to its name.")
            os.rename(zipped, wanted)
            print("Renamed to:", wanted)

        else:
            raise ValueError("Outer ZIP")

    except:
        
        with zipfile.ZipFile(zipped, "r") as z:
            print("Inside ZIP:", z.namelist())

            npz_files = [n for n in z.namelist() if n.endswith(".npz")]

            if not npz_files:
                raise RuntimeError("No .npz file found inside the ZIP.")

            z.extract(npz_files[0], PROJECT_DIR)

            extracted = os.path.join(PROJECT_DIR, npz_files[0])

            if extracted != wanted:
                os.replace(extracted, wanted)

            print("Extracted to:", wanted)

print("Final NPZ exists:", os.path.exists(wanted))

In [ ]:
# COLAB — Load EXACT original-laptop feature cache
if not os.path.exists(EXACT_FEATURE_CACHE_PATH):
    raise FileNotFoundError(
        "Missing exact_v2_feature_cache_2249.npz in MyDrive/DDI_V2. "
        "Run Export_Exact_V2_Features_Original_Laptop.ipynb on the original laptop first, "
        "then upload the generated NPZ and JSON files to MyDrive/DDI_V2."
    )

if not os.path.exists(EXACT_FEATURE_META_PATH):
    raise FileNotFoundError(
        "Missing exact_v2_feature_meta.json in MyDrive/DDI_V2."
    )

exact_cache = np.load(EXACT_FEATURE_CACHE_PATH, allow_pickle=True)
x_morgan_rdkit = exact_cache["x"].astype(np.float32)
cached_all_drugs = exact_cache["all_drugs"].tolist()

with open(EXACT_FEATURE_META_PATH, "r", encoding="utf-8") as f:
    feature_meta = json.load(f)

print("Exact cached feature shape:", x_morgan_rdkit.shape)
print("Exact cache RDKit version:", feature_meta.get("rdkit_version"))
print("Exact cache kept RDKit descriptors:", feature_meta.get("rdkit_kept_count"))

# Verify the exact node ordering matches the current split files
if cached_all_drugs != all_drugs:
    raise RuntimeError(
        "Drug ordering differs between exact feature cache and current split files. "
        "Do not train because node features would map to the wrong drugs."
    )

assert x_morgan_rdkit.shape == (EXPECTED_UNIQUE_DRUGS, EXPECTED_FEATURE_DIM)
assert feature_meta["rdkit_original_count"] == 208
assert feature_meta["rdkit_kept_count"] == EXPECTED_RDKIT_KEPT

print("Exact original V2 features loaded and validated ")


In [ ]:
# COLAB STEP 12 — Dataset and loaders
class DDIPairDataset(Dataset):
    def __init__(self, df, drug_to_id):
        self.d1 = df["Drug1"].map(drug_to_id).values.astype(np.int64)
        self.d2 = df["Drug2"].map(drug_to_id).values.astype(np.int64)
        self.y = df["label_id"].values.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.d1[idx], dtype=torch.long),
            torch.tensor(self.d2[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long)
        )

train_ds = DDIPairDataset(df_train, drug_to_id)
cal_ds = DDIPairDataset(df_cal, drug_to_id)
test_ds = DDIPairDataset(df_test, drug_to_id)

def make_loaders(seed):
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        generator=g, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    cal_loader = DataLoader(
        cal_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    return train_loader, cal_loader, test_loader

In [ ]:
# COLAB STEP 13 — Exact class weights
class_counts = np.bincount(df_train["label_id"].values, minlength=num_classes)

class_weights = np.sqrt(class_counts.max() / (class_counts + 1))
class_weights = np.clip(class_weights, 0.5, 3.0)
class_weights = class_weights / class_weights.mean()

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Min weight:", class_weights.min())
print("Max weight:", class_weights.max())

In [ ]:
# COLAB STEP 14 — Exact V2 architecture
class GCNDrugEncoderV2(nn.Module):
    def __init__(self, in_dim, hidden_dim=384, emb_dim=192, dropout=0.35):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, emb_dim)
        self.bn3 = nn.BatchNorm1d(emb_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        return x

class DDIEdgeClassifierV2(nn.Module):
    def __init__(self, in_dim, num_classes, hidden_dim=512, emb_dim=192, dropout=0.35):
        super().__init__()

        self.encoder = GCNDrugEncoderV2(
            in_dim=in_dim,
            hidden_dim=GCN_HIDDEN_DIM,
            emb_dim=emb_dim,
            dropout=dropout
        )

        pair_dim = emb_dim * 4

        self.classifier = nn.Sequential(
            nn.LayerNorm(pair_dim),
            nn.Linear(pair_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, graph_data, d1, d2):
        z = self.encoder(graph_data.x, graph_data.edge_index)
        z1 = z[d1]
        z2 = z[d2]

        pair = torch.cat([
            z1, z2, torch.abs(z1-z2), z1*z2
        ], dim=1)

        return self.classifier(pair)

In [ ]:
# COLAB STEP 15 — Evaluation
def evaluate_model(model, graph_data, loader, device):
    model.eval()
    all_y, all_pred, all_probs = [], [], []

    with torch.no_grad():
        for d1, d2, y in loader:
            d1 = d1.to(device)
            d2 = d2.to(device)

            logits = model(graph_data, d1, d2)
            probs = torch.softmax(logits, dim=1)

            all_y.append(y.cpu().numpy())
            all_pred.append(probs.argmax(dim=1).cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    probs = np.vstack(all_probs)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs,
    }

In [ ]:
# COLAB STEP 16 — Safe checkpoint helpers
def save_atomic_torch(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_history_atomic(history, path):
    tmp = path + ".tmp"
    pd.DataFrame(history).to_csv(tmp, index=False)
    os.replace(tmp, path)

def make_payload(model, optimizer, scheduler, epoch, best_cal_macro, best_epoch, wait, history, seed, completed=False):
    return {
        "model_state": copy.deepcopy(model.state_dict()),
        "optimizer_state": copy.deepcopy(optimizer.state_dict()),
        "scheduler_state": copy.deepcopy(scheduler.state_dict()),
        "epoch": int(epoch),
        "best_cal_macro": float(best_cal_macro),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "wait": int(wait),
        "history": copy.deepcopy(history),
        "seed": int(seed),
        "completed": bool(completed),
        "feature_dim": int(x_morgan_rdkit.shape[1]),
        "num_classes": int(num_classes),
        "label_names": label_names,
        "input_checksums": input_checksums,
    }

In [ ]:
# COLAB STEP 17 — Auto-save + auto-resume training
def train_v2_auto_resume(exp_name="V2_WEIGHTED_GCN_FIXED_AUTO_RESUME", seed=42):
    set_seed(seed)

    train_loader, cal_loader, test_loader = make_loaders(seed)

    graph_data = Data(
        x=torch.tensor(x_morgan_rdkit, dtype=torch.float32),
        edge_index=edge_index
    ).to(device)

    model = DDIEdgeClassifierV2(
        in_dim=x_morgan_rdkit.shape[1],
        num_classes=num_classes,
        hidden_dim=CLASSIFIER_HIDDEN_DIM,
        emb_dim=EMB_DIM,
        dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    start_epoch = 1
    best_cal_macro = -np.inf
    best_epoch = None
    wait = 0
    history = []

    if AUTO_RESUME and os.path.exists(LAST_CKPT_PATH):
        ckpt = torch.load(LAST_CKPT_PATH, map_location=device, weights_only=False)

        if ckpt.get("input_checksums") != input_checksums:
            raise RuntimeError("Checkpoint does not match current data files.")

        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_cal_macro = float(ckpt["best_cal_macro"])
        best_epoch = ckpt["best_epoch"]
        wait = int(ckpt["wait"])
        history = list(ckpt.get("history", []))

        print(f" Resuming from epoch {start_epoch}")
        print("Best epoch so far:", best_epoch)
        print("Best CAL Macro so far:", best_cal_macro)
        print("Wait:", wait)

        if ckpt.get("completed", False):
            print("Training already marked completed.")
            return {"status": "already_completed"}

    else:
        print("Starting fresh from epoch 1.")

    try:
        for epoch in range(start_epoch, MAX_EPOCHS + 1):
            model.train()
            total_loss = 0.0
            t0 = time.time()

            for d1, d2, y in tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}", leave=True):
                d1 = d1.to(device)
                d2 = d2.to(device)
                y = y.to(device)

                optimizer.zero_grad(set_to_none=True)
                logits = model(graph_data, d1, d2)
                loss = criterion(logits, y)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)

            cal_res = evaluate_model(model, graph_data, cal_loader, device)
            scheduler.step(cal_res["macro_f1"])
            lr = optimizer.param_groups[0]["lr"]

            improved = cal_res["macro_f1"] > best_cal_macro

            if improved:
                best_cal_macro = float(cal_res["macro_f1"])
                best_epoch = epoch
                wait = 0
            else:
                wait += 1

            row = {
                "experiment": exp_name,
                "seed": seed,
                "epoch": epoch,
                "loss": avg_loss,
                "cal_accuracy": float(cal_res["accuracy"]),
                "cal_macro_f1": float(cal_res["macro_f1"]),
                "cal_weighted_f1": float(cal_res["weighted_f1"]),
                "lr": float(lr),
                "wait": int(wait),
                "improved": bool(improved),
                "time_min": (time.time()-t0)/60
            }
            history.append(row)

            save_history_atomic(history, HISTORY_PATH)

            if improved:
                save_atomic_torch(
                    make_payload(model, optimizer, scheduler, epoch, best_cal_macro, best_epoch, wait, history, seed),
                    BEST_CKPT_PATH
                )
                print(" BEST checkpoint saved.")

            save_atomic_torch(
                make_payload(model, optimizer, scheduler, epoch, best_cal_macro, best_epoch, wait, history, seed),
                LAST_CKPT_PATH
            )

            print(
                f"Epoch {epoch:03d} | Loss={avg_loss:.4f} | "
                f"Cal Acc={cal_res['accuracy']:.4f} | "
                f"Cal Macro={cal_res['macro_f1']:.4f} | "
                f"Cal Weighted={cal_res['weighted_f1']:.4f} | "
                f"LR={lr:.1e} | Wait={wait}/{EARLY_STOP_PATIENCE} | "
                f"BestEpoch={best_epoch} | BestMacro={best_cal_macro:.4f}"
            )
            print(" LAST checkpoint saved.")

            if wait >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch}. Best epoch={best_epoch}")
                break

    except KeyboardInterrupt:
        last_ep = history[-1]["epoch"] if history else 0
        print("\n⏸ Training interrupted manually.")
        print("Last fully completed saved epoch:", last_ep)
        print("Run this cell again later to resume automatically.")
        return {
            "status": "interrupted",
            "last_completed_epoch": last_ep,
            "best_epoch": best_epoch,
            "best_cal_macro": best_cal_macro
        }

    best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state"])

    cal_final = evaluate_model(model, graph_data, cal_loader, device)
    test_final = evaluate_model(model, graph_data, test_loader, device)

    gap = abs(cal_final["macro_f1"] - float(best_ckpt["best_cal_macro"]))

    print("\nFINAL RESULT FROM TRUE BEST CHECKPOINT")
    print("Best epoch:", best_ckpt["best_epoch"])
    print("Saved best CAL Macro:", best_ckpt["best_cal_macro"])
    print("Reloaded CAL Macro:", cal_final["macro_f1"])
    print("Checkpoint CAL Macro gap:", gap)
    print("TEST Acc:", test_final["accuracy"])
    print("TEST Macro:", test_final["macro_f1"])
    print("TEST Weighted:", test_final["weighted_f1"])

    pred_df = df_test.copy()
    pred_df["pred_label_id"] = test_final["y_pred"]
    pred_df["pred_label_name"] = [label_names[i] for i in test_final["y_pred"]]
    pred_df["correct"] = pred_df["label_id"].values == pred_df["pred_label_id"].values
    pred_df.to_csv(os.path.join(SAVE_DIR, "final_test_predictions.csv"), index=False)

    result = {
        "experiment": exp_name,
        "seed": seed,
        "best_epoch": int(best_ckpt["best_epoch"]),
        "cal_accuracy": float(cal_final["accuracy"]),
        "cal_macro_f1": float(cal_final["macro_f1"]),
        "cal_weighted_f1": float(cal_final["weighted_f1"]),
        "test_accuracy": float(test_final["accuracy"]),
        "test_macro_f1": float(test_final["macro_f1"]),
        "test_weighted_f1": float(test_final["weighted_f1"]),
        "checkpoint_gap": float(gap)
    }

    pd.DataFrame([result]).to_csv(os.path.join(SAVE_DIR, "final_result.csv"), index=False)

    last_completed_epoch = history[-1]["epoch"]
    save_atomic_torch(
        make_payload(model, optimizer, scheduler, last_completed_epoch, best_cal_macro, best_epoch, wait, history, seed, completed=True),
        LAST_CKPT_PATH
    )

    return result

In [ ]:
import os

folder = "/content/drive/MyDrive/DDI_V2/v2_resume_outputs"

for f in os.listdir(folder):
    print(f)

In [ ]:
import os
import json

def prepare_seed_run(seed):
    global SAVE_DIR
    global LAST_CKPT_PATH
    global BEST_CKPT_PATH
    global HISTORY_PATH

    SAVE_DIR = os.path.join(
        PROJECT_DIR,
        "reproducibility_runs",
        f"seed_{seed}"
    )

    os.makedirs(SAVE_DIR, exist_ok=True)

    LAST_CKPT_PATH = os.path.join(SAVE_DIR, "LAST_checkpoint.pt")
    BEST_CKPT_PATH = os.path.join(SAVE_DIR, "BEST_checkpoint.pt")
    HISTORY_PATH   = os.path.join(SAVE_DIR, "training_history.csv")

    # Save environment/checksums for this seed too
    with open(
        os.path.join(SAVE_DIR, "environment.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(environment_info, f, indent=2, ensure_ascii=False)

    with open(
        os.path.join(SAVE_DIR, "input_checksums.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(input_checksums, f, indent=2, ensure_ascii=False)

    print("Seed:", seed)
    print("SAVE_DIR:", SAVE_DIR)
    print("LAST:", LAST_CKPT_PATH)
    print("BEST:", BEST_CKPT_PATH)

In [ ]:
prepare_seed_run(43)

result_43 = train_v2_auto_resume(
    exp_name="V2_WEIGHTED_GCN_SEED_43",
    seed=43
)

print(result_43)